In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KGramConstruction") \
    .getOrCreate()

sc = spark.sparkContext

26/03/04 17:05:26 WARN Utils: Your hostname, rajesh-pc resolves to a loopback address: 127.0.1.1; using 192.168.0.39 instead (on interface wlp1s0)
26/03/04 17:05:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 17:05:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/04 17:05:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/04 17:05:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/04 17:05:32 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
import random
import hashlib
from itertools import combinations

In [3]:
base_path = "/home/rajesh/CSL7100/Assignment2/minhash/"

files = [
    base_path + "D1.txt",
    base_path + "D2.txt",
    base_path + "D3.txt",
    base_path + "D4.txt"
]

In [4]:
docs = {}

for i, file in enumerate(files):
    with open(file, "r") as f:
        docs[f"D{i+1}"] = f.read().strip()


In [5]:
print("D1: " + docs["D1"][:50])
print("D2: " + docs["D2"][:50])
print("D3: " + docs["D3"][:50])
print("D4: " + docs["D4"][:50])


D1: apple ceo tim cook is spending some time in canada
D2: apple ceo tim cook is spending some time in canada
D3: as part of his one day tour of canada yesterday ti
D4: president trump who warned as a candidate about th


## BUILD ALL k-GRAMS ONCE (Question 1)

In [6]:
def char_kgrams(text, k):
    text = text.strip()        # remove leading/trailing spaces
    result = set()             # use set to remove duplicates

    for i in range(len(text) - k + 1):   # valid start positions
        gram = text[i:i+k]               # substring of length k
        result.add(gram)                 # add to set (no duplicates)

    return result

In [7]:
def word_2grams(text):
    words = text.strip().split()   # split into words
    result = set()                 # use set to remove duplicates

    for i in range(len(words) - 1):   # until second-last word
        gram = words[i] + " " + words[i+1]   # combine two words
        result.add(gram)                   # add to set

    return result

In [8]:
char2_sets = {}
char3_sets = {}
word2_sets = {}

for name, text in docs.items():

    # Character 2-grams (set removes duplicates)
    char2_sets[name] = char_kgrams(text,2)

    # Character 3-grams
    char3_sets[name] = char_kgrams(text,3)

    # Word 2-grams
    words = text.split()
    word2_sets[name] = word_2grams(text)

In [9]:
print(list(char2_sets["D1"])[:5])
print(list(char3_sets["D2"])[:5])
print(list(word2_sets["D2"])[:5])

['rm', 'lc', 'sm', 'wo', 'eg']
['try', 's m', 'aid', 'rm ', 'o d']
['very immersive', 'apps and', 'quality as', 'than competing', 'interview which']


In [10]:
# Use already created 3-gram sets
S1 = char3_sets["D1"]
S2 = char3_sets["D2"]

In [11]:
# Compute true Jaccard similarity between two sets
def true_jaccard(set1, set2):
    intersection = set1 & set2   # common elements
    union = set1 | set2          # all unique elements
    return len(intersection) / len(union)   # J(A,B) = |A∩B| / |A∪B|


In [12]:
import itertools

# LSH parameters
t = 160      # total hash functions
r = 20       # number of bands
b = 8        # rows per band (r*b = t)
tau = 0.7    # threshold

# S-curve probability function
def lsh_probability(s, r, b):
    return 1 - (1 - s**b)**r

# List of documents
doc_names = ["D1","D2","D3","D4"]

# Compute all 6 pairs
pairs = list(itertools.combinations(doc_names, 2))

print("Pair\tJaccard\tLSH Probability (s > τ)")
for (i,j) in pairs:
    # Compute actual Jaccard using your previous function
    s = true_jaccard(char3_sets[i], char3_sets[j])
    
    # Compute probability of LSH retrieving pair
    p = lsh_probability(s, r, b)
    
    print(f"{i}-{j}\t{s:.4f}\t{p:.4f}")

Pair	Jaccard	LSH Probability (s > τ)
D1-D2	0.9780	1.0000
D1-D3	0.5804	0.2282
D1-D4	0.3051	0.0015
D2-D3	0.5680	0.1959
D2-D4	0.3059	0.0015
D3-D4	0.3121	0.0018


In [14]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 18.0 MB/s  0:00:00 eta 0:00:01:01:01
Note: you may need to restart the kernel to use updated packages.


In [15]:
import itertools
import pandas as pd

# Total hash functions
t = 160

# Different (r, b) configurations (must satisfy r * b = t)
configs = [
    (20, 8),
    (40, 4),
    (10, 16),
    (80, 2),
    (5, 32)
]

# LSH probability function
def lsh_probability(s, r, b):
    return 1 - (1 - s**b)**r

# Document names
doc_names = ["D1", "D2", "D3", "D4"]
pairs = list(itertools.combinations(doc_names, 2))

print("Pair\tJaccard\t" + "\t".join([f"r={r},b={b}" for (r,b) in configs]))

for (i, j) in pairs:
    
    s = true_jaccard(char3_sets[i], char3_sets[j])
    
    row_output = f"{i}-{j}\t{s:.4f}"
    
    for (r, b) in configs:
        p = lsh_probability(s, r, b)
        row_output += f"\t{p:.4f}"
    
    print(row_output)

Pair	Jaccard	r=20,b=8	r=40,b=4	r=10,b=16	r=80,b=2	r=5,b=32
D1-D2	0.9780	1.0000	1.0000	1.0000	1.0000	0.9656
D1-D3	0.5804	0.2282	0.9919	0.0017	1.0000	0.0000
D1-D4	0.3051	0.0015	0.2939	0.0000	0.9996	0.0000
D2-D3	0.5680	0.1959	0.9877	0.0012	1.0000	0.0000
D2-D4	0.3059	0.0015	0.2966	0.0000	0.9996	0.0000
D3-D4	0.3121	0.0018	0.3171	0.0000	0.9997	0.0000


In [18]:
import itertools
import pandas as pd

# Total hash functions
t = 160

# Different (r, b) configurations (must satisfy r * b = t)
configs = [
    (20, 8),
    (40, 4),
    (10, 16),
    (80, 2),
    (5, 32)
]

# LSH probability function
def lsh_probability(s, r, b):
    return 1 - (1 - s**b)**r

# Document names
doc_names = ["D1", "D2", "D3", "D4"]
pairs = list(itertools.combinations(doc_names, 2))

print("Pair\tJaccard\t" + "\t".join([f"r={r},b={b}" for (r,b) in configs]))

for (i, j) in pairs:
    
    s = true_jaccard(char3_sets[i], char3_sets[j])
    
    row_output = f"{i}-{j}\t{s:.4f}"
    
    for (r, b) in configs:
        p = lsh_probability(s, r, b)
        row_output += f"\t{p:.4f}"
    
    print(row_output)

Pair	Jaccard	r=20,b=8	r=40,b=4	r=10,b=16	r=80,b=2	r=5,b=32
D1-D2	0.9780	1.0000	1.0000	1.0000	1.0000	0.9656
D1-D3	0.5804	0.2282	0.9919	0.0017	1.0000	0.0000
D1-D4	0.3051	0.0015	0.2939	0.0000	0.9996	0.0000
D2-D3	0.5680	0.1959	0.9877	0.0012	1.0000	0.0000
D2-D4	0.3059	0.0015	0.2966	0.0000	0.9996	0.0000
D3-D4	0.3121	0.0018	0.3171	0.0000	0.9997	0.0000


In [58]:
if spark:
    spark.stop()